# Experiment 4.0.6 — Raw64 local membrane-memory ablation

Analysis-only notebook. Training is performed by the Slurm arrays documented in `scripts/experiment_4_0_6/README.md`.

Primary question: can stateful Local128 membrane integration recover the Raw64 information that was poorly expressed when `beta_local=0`, without explicit Fixed250 pooling?

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_4_0_6_local_mem_raw64' / 'raw64_local_mem_shift_ablation_v1'
summary = pd.read_csv(ART / 'summary.csv')
effects = pd.read_csv(ART / 'paired_effects_summary.csv')
summary

## Main test BA table
Compare the frozen `beta0` Raw64 baseline against `shift4`, `shift34`, and `shift234` for all three readouts.

In [ ]:
cols = [
    'condition', 'variant',
    'valid_count_balanced_accuracy_mean', 'valid_count_balanced_accuracy_std',
    'hidden_count_linear_balanced_accuracy_mean', 'hidden_count_linear_balanced_accuracy_std',
    'uend_linear_balanced_accuracy_mean', 'uend_linear_balanced_accuracy_std',
]
summary[cols].sort_values(['variant', 'condition'])

## Local-memory effect relative to beta0
The primary effects are `shift4_minus_beta0`, `shift34_minus_beta0`, and `shift234_minus_beta0`.

In [ ]:
primary = effects[effects['effect'].isin([
    'shift4_minus_beta0',
    'shift34_minus_beta0',
    'shift234_minus_beta0',
])].copy()
primary.sort_values(['readout', 'variant', 'effect'])

In [ ]:
for readout in ['output_whole_count', 'hidden_whole_count_linear', 'uend_linear']:
    view = primary[primary['readout'] == readout].copy()
    pivot = view.pivot(index='variant', columns='effect', values='mean')
    ax = pivot.plot(kind='bar', figsize=(9, 5))
    ax.axhline(0.0, linewidth=1)
    ax.set_ylabel('Paired test BA delta')
    ax.set_title(readout)
    plt.tight_layout()
    plt.show()

## Dynamics diagnostics
Use physical event rates rather than events/timestep because all conditions run Raw64 but the Local membrane distributions differ. Check whether improvements track useful state formation rather than saturation/cap hits.

In [ ]:
diag_cols = [
    'condition', 'variant',
    'local_mean_events_per_neuron_second_mean',
    'state_mean_events_per_neuron_second_mean',
    'output_mean_events_per_neuron_second_mean',
    'local_fraction_at_cap_mean',
    'state_fraction_at_cap_mean',
    'output_fraction_at_cap_mean',
    'state_tail_event_fraction_mean',
    'output_tail_event_fraction_mean',
]
summary[diag_cols].sort_values(['variant', 'condition'])